In [ ]:
print("Hello from AI Cup")

In [ ]:
!echo hello

In [ ]:
import numpy as np

In [ ]:
# import skyfield

# Let's load the data

In [ ]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/ai-cup-2026-performance/train.csv")
test_df = pd.read_csv("/kaggle/input/ai-cup-2026-performance/test.csv")

train_df = train_df.set_index("track_id")
test_df = test_df.set_index("track_id")

train_df

# Let's explore the data

In [ ]:
import matplotlib.pyplot as plt

train_df['bird_group'].value_counts().plot.pie(autopct="%1.1f%%")
plt.show()

In [ ]:
from shapely import wkb

wkb_hex = train_df['trajectory'].iloc[0]
 
geom = wkb.loads(bytes.fromhex(wkb_hex))
coords = list(geom.coords)

coords

In [ ]:
def split_xyzm(wkb_hex):
    geom = wkb.loads(bytes.fromhex(wkb_hex))
    coords = list(geom.coords)

    xs, ys, zs, RCSs = zip(*coords)

    return pd.Series({
        "x": xs,
        "y":ys,
        "z":zs,
        "RCS":RCSs
    })

extra_train_cols = train_df['trajectory'].apply(split_xyzm)
extra_test_cols = test_df['trajectory'].apply(split_xyzm)

train_df = train_df.join(extra_train_cols)

test_df = test_df.join(extra_test_cols)

train_df['mean_RCS'] = train_df['RCS'].apply(np.mean)
test_df['mean_RCS'] = test_df['RCS'].apply(np.mean)

test_df

In [ ]:
import seaborn as sns

num_cols_to_plot = ['airspeed', 'max_z', 'mean_RCS']

for feat in num_cols_to_plot:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=train_df, x='bird_group', y=feat, palette='viridis', hue='bird_group')
    plt.title(f"{feat} per bird_group")
    plt.xticks(rotation=45)
    plt.show()
    plt.close()

# Prepare features


In [ ]:
#model input X
#model targets y

features = ['airspeed', 'min_z', 'max_z', 'mean_RCS', 'radar_bird_size']

X = train_df[features]
X_test = test_df[features]

y = train_df['bird_group']

X

print(X_test)

In [ ]:
y

# Setup the pipeline


In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


preprocessor = ColumnTransformer(
    [('cat', OneHotEncoder(), ['radar_bird_size'])],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

# Cross validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 5

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

split = list(cv.split(X, y))

for i, (train_idx, val_idx) in enumerate(split):
    print(f"fold {i+1} train: {len(train_idx)}, val:  {len(val_idx)}" )

# Training the model


In [ ]:
from sklearn.base import clone

classes = np.unique(y)

oof_preds = pd.DataFrame(0.0, index=X.index, columns=classes)

test_preds = np.zeros((len(X_test), len(classes)))

for i, (train_idx, val_idx) in enumerate(split):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    pipeline_fold = clone(pipeline)

    pipeline_fold.fit(X_train, y_train)

    val_preds = pipeline_fold.predict_proba(X_val)
    oof_preds.iloc[val_idx] = val_preds

    test_preds_fold = pipeline_fold.predict_proba(X_test)

    test_preds += test_preds_fold

    print(f"Trained fold{i+1}/{n_splits}!")
    

In [ ]:
oof_preds

In [ ]:
import pandas as pd
import sklearn.metrics

def score(
    solution: pd.DataFrame, 
    submission: pd.DataFrame, 
) -> float:
    #Takes in two pandas dataframe and computes the Macro-averaged Average Precision Score
   
    # Ensure all required columns are present
    needed_columns = [
        "Clutter", 
        "Cormorants", 
        "Pigeons", 
        "Ducks", 
        "Geese", 
        "Gulls", 
        "Birds of Prey", 
        "Waders", 
        "Songbirds",
    ]
    
    # Reorder solution and submission columns/rows to match exactly
    solution = solution.loc[solution.index, needed_columns]
    submission = submission.loc[solution.index, needed_columns]

    
    # Compute the Average Precision score for all required columns
    bird_score = sklearn.metrics.average_precision_score(
        solution[needed_columns],
        submission[needed_columns],
        average='macro'
    )

    return bird_score

In [ ]:
# Get local OOF CV score
solution_df = (
    train_df
    .groupby(["track_id", "bird_group"])
    .size()
    .unstack(fill_value=0)
)

oof_score = score(solution_df, oof_preds)

print(f"OOF score: {oof_score}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Convert OOF probabilities to hard class labels
y_pred_oof = classes[np.argmax(oof_preds, axis=1)]

# Plot Normalized Confusion Matrix
fig, ax = plt.subplots(figsize=(12, 10))

ConfusionMatrixDisplay.from_predictions(
    y, 
    y_pred_oof, 
    display_labels=classes,
    cmap="Blues",
    normalize='true',
    values_format=".2f",
    xticks_rotation=45,
    ax=ax
)

plt.title("Normalized Confusion Matrix (OOF)")
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
submission_df = pd.DataFrame(
    test_preds / n_splits,
    index=X_test.index,
    columns=classes
)

submission_df.to_csv('submission.csv')

print("Saved to submission.csv")

!head submission.csv